Рубежный контроль №1
Дисциплина: Интеллектуальный анализ данных / технологии разведочного анализа и обработки данных
Тема: Обработка пропусков в данных и сравнение исходного и итогового наборов данных
Студент: Попов Дмитрий Денисович Группа: ИБМ3-65Б
Задача №2.
Для заданного набора данных проведите обработку пропусков в данных для одного категориального и одного количественного признака. Какие способы обработки пропусков в данных для категориальных и количественных признаков Вы использовали? Какие признаки Вы будете использовать для дальнейшего построения моделей машинного обучения и почему?

В работе используется набор данных Admission_Predict_Ver1.1 . Для выполнения задания были искусственно добавлены категориальные признаки и пропуски, после чего выполнена обработка пропусков:

для количественных признаков применяется метод MICE (IterativeImputer);
для категориальных признаков используется заполнение наиболее частым значением (most_frequent), после чего выполняется Ordinal Encoding.
Дополнительно в отчете построена гистограмма для одного из признаков и выполнено сравнение начального и итогового датасетов.
14	2	6

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

# Настройки отображения
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Библиотеки успешно импортированы!")

In [ ]:
## 1. Загрузка и первичный анализ данных

In [ ]:
# Загрузка датасета
df = pd.read_csv('Admission_Predict_Ver1.1.csv')

# Просмотр первых строк
display(df.head(10))

# Основная информация о датасете
print("=== Информация о датасете ===")
df.info()

# Статистическое описание
print("\n=== Статистическое описание ===")
display(df.describe())

# Проверка на пропущенные значения
print("\n=== Пропущенные значения ===")
missing = df.isnull().sum()
print(missing)
print(f"\nВсего пропусков в датасете: {missing.sum()}")

In [ ]:
**Выводы по первичному анализу:**
- Датасет содержит 500 наблюдений и 9 признаков.
- Все столбцы имеют числовой тип данных (int64/float64).
- **Пропущенных значений в исходных данных нет** (все 0).
- Признаки:
  - `Serial No.` — идентификатор (неинформативен для модели).
  - `GRE Score`, `TOEFL Score`, `CGPA`, `Chance of Admit` — **количественные**.
  - `University Rating`, `SOP`, `LOR`, `Research` — **категориальные/ординальные** (шкала 1–5 или 0/1).

Для демонстрации обработки пропусков **искусственно введём** пропуски в один категориальный и один количественный признак.

In [ ]:
# Создаём копию для демонстрации обработки пропусков
df_demo = df.copy()

# Искусственно вводим пропуски (10% случайных значений)
np.random.seed(42)
n_missing = int(len(df_demo) * 0.1)

# 1. Категориальный признак: University Rating
cat_feature = 'University Rating'
missing_idx_cat = np.random.choice(df_demo.index, n_missing, replace=False)
df_demo.loc[missing_idx_cat, cat_feature] = np.nan

# 2. Количественный признак: CGPA
quant_feature = 'CGPA'
missing_idx_quant = np.random.choice(df_demo.index, n_missing, replace=False)
df_demo.loc[missing_idx_quant, quant_feature] = np.nan

print(f"Искусственно добавлено пропусков в '{cat_feature}': {df_demo[cat_feature].isnull().sum()}")
print(f"Искусственно добавлено пропусков в '{quant_feature}': {df_demo[quant_feature].isnull().sum()}")

In [ ]:
## 2. Обработка пропусков

In [ ]:
# Визуализация пропусков до обработки
plt.figure(figsize=(10, 4))
sns.heatmap(df_demo.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Пропуски в данных (до обработки)')
plt.show()

In [ ]:
### 2.1. Обработка категориального признака (`University Rating`)

**Выбранный способ:** заполнение **модой** (самым частым значением).

**Обоснование:**
- Категориальные признаки нельзя заполнять средним/медианой.
- Мода сохраняет распределение и не искажает смысл признака.
- Альтернативы (не использовались здесь): KNNImputer, создание новой категории "Missing", удаление строк (если мало пропусков).

In [ ]:
# Заполнение категориального признака модой
mode_value = df_demo[cat_feature].mode()[0]
df_demo[cat_feature] = df_demo[cat_feature].fillna(mode_value)

print(f"Заполнили пропуски в '{cat_feature}' значением моды = {mode_value}")
print(f"Осталось пропусков в '{cat_feature}': {df_demo[cat_feature].isnull().sum()}")

In [ ]:
# Заполнение категориального признака модой
mode_value = df_demo[cat_feature].mode()[0]
df_demo[cat_feature] = df_demo[cat_feature].fillna(mode_value)

print(f"Заполнили пропуски в '{cat_feature}' значением моды = {mode_value}")
print(f"Осталось пропусков в '{cat_feature}': {df_demo[cat_feature].isnull().sum()}")

In [ ]:
# Визуализация после обработки
plt.figure(figsize=(10, 4))
sns.heatmap(df_demo.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Пропуски в данных (после обработки)')
plt.show()